<a href="https://colab.research.google.com/github/Yassminefeki/brain-tumor-mlops/blob/main/notebooks/02_colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Colab runner (GPU training)

This notebook only **launches** code from the repository: all logic lives in `src/brain_tumor/`.

**Runtime → Change runtime type → GPU (T4)**, then *Run all*.

Colab secrets (key icon in the left bar, enable *Notebook access* for each):

| Secret | Needed? | Used for |
|---|---|---|
| `GITHUB_TOKEN` | **yes** while the repository is private | cloning the repo (and pushing tuned parameters) |
| `DAGSHUB_USER`, `DAGSHUB_TOKEN` | optional | remote MLflow server + `dvc pull` from DagsHub |

Without the DagsHub secrets the notebook still works: it rebuilds the dataset from Figshare with
`dvc repro` (~5 min), logs MLflow runs to a local `mlflow.db`, and the last section copies the
checkpoints, reports and MLflow database to your Google Drive so nothing is lost when the
session ends.

In [4]:
# ---- Settings ----
GITHUB_USER = "Yassminefeki"
REPO = "brain-tumor-mlops"
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/brain-tumor-mlops-results"
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Credentials

In [5]:
import os
from google.colab import userdata


def secret(name):
    try:
        return userdata.get(name) or None
    except Exception:  # secret missing or notebook access not granted
        return None


GITHUB_TOKEN = secret("GITHUB_TOKEN")
DAGSHUB_USER, DAGSHUB_TOKEN = secret("DAGSHUB_USER"), secret("DAGSHUB_TOKEN")
USE_DAGSHUB = bool(DAGSHUB_USER and DAGSHUB_TOKEN)

if USE_DAGSHUB:
    os.environ.update({
        "MLFLOW_TRACKING_URI": f"https://dagshub.com/{DAGSHUB_USER}/{REPO}.mlflow",
        "MLFLOW_TRACKING_USERNAME": DAGSHUB_USER,
        "MLFLOW_TRACKING_PASSWORD": DAGSHUB_TOKEN,
    })
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
print("GitHub token:", "yes" if GITHUB_TOKEN else "no (public repo only)")
print("MLflow:", "DagsHub" if USE_DAGSHUB else "local mlflow.db (saved to Drive at the end)")

GitHub token: yes
MLflow: local mlflow.db (saved to Drive at the end)


## 2. Get the code and install it

In [6]:
auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
if not os.path.exists(f"/content/{REPO}"):
    !git clone -q -b {BRANCH} https://{auth}github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git pull -q
!pip install -q -e .
!git log -1 --oneline

/content/brain-tumor-mlops
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for brain-tumor-mlops (pyproject.toml) ... done
8aa5f91 (HEAD -> main, origin/main, origin/HEAD) docs: keep the outputs of the EDA notebook


## 3. Data

With DagsHub: pull the exact versioned data (~300 MB). Without it: rebuild it from Figshare;
`dvc.lock` in the repository guarantees the result is the same data version.

In [7]:
remotes = !dvc remote list
has_remote = USE_DAGSHUB and any(line.startswith("origin") for line in remotes)
if has_remote:
    !dvc remote modify origin --local auth basic
    !dvc remote modify origin --local user {DAGSHUB_USER}
    !dvc remote modify origin --local password {DAGSHUB_TOKEN}
    !dvc pull data/processed data/splits.csv data/metadata.csv
else:
    !dvc repro
!python -m brain_tumor.data.validate

Running stage 'download':
> python -m brain_tumor.data.ingest download
20:16:39 | INFO    | brain_tumor.ingest | Downloading brainTumorDataPublic_1-766.zip (204.5 MiB)
20:16:43 | INFO    | brain_tumor.ingest | Downloading brainTumorDataPublic_1533-2298.zip (205.6 MiB)
20:16:46 | INFO    | brain_tumor.ingest | Downloading brainTumorDataPublic_767-1532.zip (207.8 MiB)
20:16:48 | INFO    | brain_tumor.ingest | Downloading brainTumorDataPublic_2299-3064.zip (220.9 MiB)
20:16:50 | INFO    | brain_tumor.ingest | Downloading cvind.mat (0.0 MiB)
20:16:51 | INFO    | brain_tumor.ingest | Downloading README 2024.txt (0.0 MiB)
20:16:51 | INFO    | brain_tumor.ingest | Downloaded 6 files to /content/brain-tumor-mlops/data/raw
Committing data/raw to cache: 100% 7/7 [00:17<00:00,  2.52s/file{'info': ''}]
memory://.LqUxaIB0xzihB0XbTBMeCw.tmp:   0% 0.00/588 [00:00<?, ?B/s]
memory://.LqUxaIB0xzihB0XbTBMeCw.tmp:   0% 0.00/588 [00:00<?, ?B/s{'info': ''}]
Updating lock file 'dvc.lock'

Running stage 'extr

## 4. Experiments

Run the cells you need. Every run is logged to MLflow with its parameters, git commit,
data version, curves, confusion matrix, Grad-CAM and checkpoint.

In [8]:
# 0. Majority-class baseline
!python -m brain_tumor.baseline

20:19:48 | INFO    | brain_tumor.baseline | Majority class: glioma | test accuracy 0.448 | test macro-F1 0.206
2026/11/27 20:19:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/27 20:19:50 INFO mlflow.store.db.utils: Updating database tables
2026/11/27 20:19:53 INFO mlflow.tracking.fluent: Experiment with name 'brain-tumor/baseline' does not exist. Creating a new experiment.
20:19:53 | INFO    | brain_tumor.tracking | MLflow tracking: sqlite:////content/brain-tumor-mlops/mlflow.db | experiment: brain-tumor/baseline


In [9]:
# 1-2. Custom CNN without / with augmentation
!python -m brain_tumor.train train.model=custom_cnn train.augment=false
!python -m brain_tumor.train train.model=custom_cnn

20:20:43 | INFO    | brain_tumor.train | Training custom_cnn on cuda -> /content/brain-tumor-mlops/models/custom_cnn/custom_cnn-noaug-20261127-202043
20:20:45 | INFO    | brain_tumor.datamodule | Data: train 2150 | val 494 | test 420 images (split=split, fold=None)
20:20:45 | INFO    | brain_tumor.train | custom_cnn: 2.39M parameters
2026/11/27 20:20:46 INFO mlflow.tracking.fluent: Experiment with name 'brain-tumor/custom_cnn' does not exist. Creating a new experiment.
20:20:46 | INFO    | brain_tumor.tracking | MLflow tracking: sqlite:////content/brain-tumor-mlops/mlflow.db | experiment: brain-tumor/custom_cnn
20:20:46 | INFO    | brain_tumor.engine | Stage 'train': 50 epochs, lrs={'all': 0.001}
20:21:01 | INFO    | brain_tumor.engine | [custom_cnn] epoch   1 | train loss 0.955 acc 0.646 | val loss 1.078 acc 0.628 F1 0.548 | 14s
20:21:07 | INFO    | brain_tumor.engine | [custom_cnn] epoch   2 | train loss 0.730 acc 0.777 | val loss 1.447 acc 0.383 F1 0.354 | 6s
20:21:13 | INFO    | br

In [10]:
# 3-4. Pretrained models (two-stage fine-tuning)
!python -m brain_tumor.train train.model=resnet18
!python -m brain_tumor.train train.model=efficientnet_b0

20:28:00 | INFO    | brain_tumor.train | Training resnet18 on cuda -> /content/brain-tumor-mlops/models/resnet18/resnet18-aug-20261127-202800
20:28:01 | INFO    | brain_tumor.datamodule | Data: train 2150 | val 494 | test 420 images (split=split, fold=None)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 221MB/s]
20:28:02 | INFO    | brain_tumor.train | resnet18: 11.18M parameters
2026/11/27 20:28:03 INFO mlflow.tracking.fluent: Experiment with name 'brain-tumor/resnet18' does not exist. Creating a new experiment.
20:28:03 | INFO    | brain_tumor.tracking | MLflow tracking: sqlite:////content/brain-tumor-mlops/mlflow.db | experiment: brain-tumor/resnet18
20:28:04 | INFO    | brain_tumor.engine | Stage 'head': 3 epochs, lrs={'head': 0.001}
20:28:12 | INFO    | brain_tumor.engine | [resnet18] epoch   1 | train loss 1.101 acc 0.468 | val loss 0.894 acc 0.678 F1 0.684 | 8s
20

In [11]:
# 5. Leakage demonstration: same model, image-level split
!python -m brain_tumor.train train.model=resnet18 train.split_column=image_split

20:34:46 | INFO    | brain_tumor.train | Training resnet18 on cuda -> /content/brain-tumor-mlops/models/resnet18/resnet18-aug-image_split-20261127-203446
20:34:46 | INFO    | brain_tumor.datamodule | Data: train 2144 | val 460 | test 460 images (split=image_split, fold=None)
20:34:47 | INFO    | brain_tumor.train | resnet18: 11.18M parameters
20:34:48 | INFO    | brain_tumor.tracking | MLflow tracking: sqlite:////content/brain-tumor-mlops/mlflow.db | experiment: brain-tumor/resnet18
20:34:49 | INFO    | brain_tumor.engine | Stage 'head': 3 epochs, lrs={'head': 0.001}
20:34:56 | INFO    | brain_tumor.engine | [resnet18] epoch   1 | train loss 1.082 acc 0.493 | val loss 0.824 acc 0.748 F1 0.725 | 7s
20:35:03 | INFO    | brain_tumor.engine | [resnet18] epoch   2 | train loss 0.808 acc 0.708 | val loss 0.657 acc 0.854 F1 0.843 | 7s
20:35:08 | INFO    | brain_tumor.engine | [resnet18] epoch   3 | train loss 0.732 acc 0.748 | val loss 0.635 acc 0.850 F1 0.832 | 5s
20:35:08 | INFO    | brain_

## 5. (Optional) Optuna tuning — same budget for every compared model

In [13]:
!python -m brain_tumor.tune train.model=custom_cnn tune.n_trials=20
!python -m brain_tumor.train train.model=custom_cnn --best-params configs/best_params/custom_cnn.yaml --run-name custom_cnn-tuned

20:39:28 | INFO    | brain_tumor.datamodule | Data: train 2150 | val 494 | test 420 images (split=split, fold=None)
[I 2026-11-27 20:39:29,744] A new study created in RDB with name: custom_cnn-tune
2026/11/27 20:39:30 INFO mlflow.tracking.fluent: Experiment with name 'brain-tumor/custom_cnn-tuning' does not exist. Creating a new experiment.
20:39:30 | INFO    | brain_tumor.tracking | MLflow tracking: sqlite:////content/brain-tumor-mlops/mlflow.db | experiment: brain-tumor/custom_cnn-tuning
20:39:30 | INFO    | brain_tumor.engine | Stage 'train': 10 epochs, lrs={'all': 0.0003574712922600243}
20:39:42 | INFO    | brain_tumor.engine | [custom_cnn] epoch   1 | train loss 1.205 acc 0.494 | val loss 1.197 acc 0.279 F1 0.184 | 11s
20:39:48 | INFO    | brain_tumor.engine | [custom_cnn] epoch   2 | train loss 0.907 acc 0.645 | val loss 0.742 acc 0.727 F1 0.743 | 7s
20:39:56 | INFO    | brain_tumor.engine | [custom_cnn] epoch   3 | train loss 0.774 acc 0.734 | val loss 0.693 acc 0.757 F1 0.772 |

In [14]:
# Push the tuned parameters back to GitHub (needs the GITHUB_TOKEN secret)
!git config user.email "colab@users.noreply.github.com" && git config user.name "Colab runner"
!git add configs/best_params && git commit -m "exp: tuned hyperparameters" || echo "nothing to commit"
!git push -q https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:{BRANCH}

[main 6424ed7] exp: tuned hyperparameters
 1 file changed, 13 insertions(+)
 create mode 100644 configs/best_params/custom_cnn.yaml


## 6. (Optional) Bonus: 5-fold patient-level cross-validation

In [15]:
for fold in range(5):
    !python -m brain_tumor.train train.model=resnet18 train.fold={fold}

20:53:55 | INFO    | brain_tumor.train | Training resnet18 on cuda -> /content/brain-tumor-mlops/models/resnet18/resnet18-aug-fold0-20261127-205355
20:53:56 | INFO    | brain_tumor.datamodule | Data: train 2080 | val 564 | test 420 images (split=split, fold=0)
20:53:57 | INFO    | brain_tumor.train | resnet18: 11.18M parameters
20:53:58 | INFO    | brain_tumor.tracking | MLflow tracking: sqlite:////content/brain-tumor-mlops/mlflow.db | experiment: brain-tumor/resnet18
20:53:58 | INFO    | brain_tumor.engine | Stage 'head': 3 epochs, lrs={'head': 0.001}
20:54:06 | INFO    | brain_tumor.engine | [resnet18] epoch   1 | train loss 1.095 acc 0.476 | val loss 0.865 acc 0.722 F1 0.715 | 8s
20:54:14 | INFO    | brain_tumor.engine | [resnet18] epoch   2 | train loss 0.824 acc 0.699 | val loss 0.766 acc 0.759 F1 0.760 | 8s
20:54:20 | INFO    | brain_tumor.engine | [resnet18] epoch   3 | train loss 0.743 acc 0.761 | val loss 0.735 acc 0.773 F1 0.770 | 6s
20:54:20 | INFO    | brain_tumor.engine | 

## 7. Final model

Retrain the best model of section 4 (change `train.model` if it is not ResNet-18), register it
in the MLflow Model Registry and copy its checkpoint to `models/best.pt` for the API.

In [16]:
!python -m brain_tumor.train train.model=resnet18 --register --promote --run-name resnet18-final

21:08:36 | INFO    | brain_tumor.train | Training resnet18 on cuda -> /content/brain-tumor-mlops/models/resnet18/resnet18-final-20261127-210836
21:08:37 | INFO    | brain_tumor.datamodule | Data: train 2150 | val 494 | test 420 images (split=split, fold=None)
21:08:38 | INFO    | brain_tumor.train | resnet18: 11.18M parameters
21:08:39 | INFO    | brain_tumor.tracking | MLflow tracking: sqlite:////content/brain-tumor-mlops/mlflow.db | experiment: brain-tumor/resnet18
21:08:39 | INFO    | brain_tumor.engine | Stage 'head': 3 epochs, lrs={'head': 0.001}
21:08:47 | INFO    | brain_tumor.engine | [resnet18] epoch   1 | train loss 1.101 acc 0.468 | val loss 0.894 acc 0.678 F1 0.684 | 7s
21:08:54 | INFO    | brain_tumor.engine | [resnet18] epoch   2 | train loss 0.828 acc 0.680 | val loss 0.711 acc 0.798 F1 0.801 | 8s
21:09:00 | INFO    | brain_tumor.engine | [resnet18] epoch   3 | train loss 0.727 acc 0.764 | val loss 0.692 acc 0.802 F1 0.805 | 6s
21:09:00 | INFO    | brain_tumor.engine | S

## 8. Save the results to Google Drive

Copies `models/` (checkpoints, curves, confusion matrices, Grad-CAM), `reports/`, `configs/`
and the local MLflow database. On your laptop, put `models/best.pt` in the project's `models/`
folder to serve it with `make api`, and copy `mlflow.db` to browse the runs with `mlflow ui`.

In [17]:
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
!cp -r models reports configs {DRIVE_DIR}/
!test -f mlflow.db && cp mlflow.db {DRIVE_DIR}/ || true
!ls -R {DRIVE_DIR} | head -40

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/brain-tumor-mlops-results:
configs
mlflow.db
models
reports

/content/drive/MyDrive/brain-tumor-mlops-results/configs:
best_params

/content/drive/MyDrive/brain-tumor-mlops-results/configs/best_params:
custom_cnn.yaml

/content/drive/MyDrive/brain-tumor-mlops-results/models:
best.pt
custom_cnn
efficientnet_b0
resnet18

/content/drive/MyDrive/brain-tumor-mlops-results/models/custom_cnn:
custom_cnn-aug-20261127-202354
custom_cnn-noaug-20261127-202043
custom_cnn-tuned-20261127-205035

/content/drive/MyDrive/brain-tumor-mlops-results/models/custom_cnn/custom_cnn-aug-20261127-202354:
best.pt
classification_report.txt
confusion_matrix.png
gradcam.png
history.json
summary.json
test_metrics.json
test_predictions.csv
training_curves.png

/content/drive/MyDrive/brain-tumor-mlops-results/models/custom_cnn/custom_cnn-noaug-20261127-202043:
best.pt
